In [1]:
#############################################
# PROJE: Hybrid Recommender System
#############################################

# ID'si verilen kullanıcı için item-based ve user-based recomennder yöntemlerini kullanarak tahmin yapınız.
# 5 öneri user-based modelden 5 öneri de item-based modelden ele alınız ve nihai olarak 10 öneriyi 2 modelden yapınız.

#############################################
# Görev 1: Verinin Hazırlanması
#############################################


# Adım 1: Movie ve Rating veri setlerini okutunuz.
# movieId, film adı ve filmin tür bilgilerini içeren veri seti

import pandas as pd
# UserID, film adı, filme verilen oy ve zaman bilgisini içeren veri seti



movie = pd.read_csv("/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/movie.csv")
rating = pd.read_csv("/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/rating.csv")

In [2]:
movie.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
rating.head()

,userId,movieId,rating,timestamp
0,1,2,3.5,2005-04-02 23:53:47
1,1,29,3.5,2005-04-02 23:31:16
2,1,32,3.5,2005-04-02 23:33:39
3,1,47,3.5,2005-04-02 23:32:07
4,1,50,3.5,2005-04-02 23:29:40


In [5]:
# Adım 2: Rating veri setine filmlerin isimlerini ve türünü movie film setini kullanrak ekleyiniz.
# Ratingdeki kullanıcıların oy kullandıkları filmlerin sadece id'si var.
# Idlere ait film isimlerini ve türünü movie veri setinden ekliyoruz.

df = pd.merge(rating, movie, on='movieId')
df.head()

,userId,movieId,rating,timestamp,title,genres
0,1,2,3.5,2005-04-02 23:53:47,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,2005-04-02 23:31:16,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,2005-04-02 23:33:39,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,2005-04-02 23:32:07,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,2005-04-02 23:29:40,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [6]:
# Adım 3: Toplam oy kullanılma sayısı 1000'in altında olan filmlerin isimlerini listede tutunuz ve veri setinden çıkartınız.
# Eğer toplam oy kullanılma sayısı 1000'in altında olan filmleri çıkarmak isterseniz
movie_counts = df['title'].value_counts()
popular_movies = movie_counts[movie_counts >= 1000].index
filtered_df = df[df['title'].isin(popular_movies)]

# Adım 4: index'te userID'lerin sutunlarda film isimlerinin ve değer olarak ratinglerin bulunduğu dataframe için pivot table oluşturunuz.
# Pivot table oluşturun
pivot_df = filtered_df.pivot_table(index='userId', columns='title', values='rating')
print(pivot_df.head())

title   'burbs, The (1989)  (500) Days of Summer (2009)  \
userId                                                    
1                      NaN                          NaN   
2                      NaN                          NaN   
3                      NaN                          NaN   
4                      NaN                          NaN   
5                      NaN                          NaN   

title   *batteries not included (1987)  ...And Justice for All (1979)  \
userId                                                                  
1                                  NaN                            NaN   
2                                  NaN                            NaN   
3                                  NaN                            NaN   
4                                  NaN                            NaN   
5                                  NaN                            NaN   

title   10 Things I Hate About You (1999)  10,000 BC (2008)  \
userId     

In [7]:
print(pivot_df.head(20))

title   'burbs, The (1989)  (500) Days of Summer (2009)  \
userId                                                    
1                      NaN                          NaN   
2                      NaN                          NaN   
3                      NaN                          NaN   
4                      NaN                          NaN   
5                      NaN                          NaN   
6                      NaN                          NaN   
7                      NaN                          NaN   
8                      NaN                          NaN   
9                      NaN                          NaN   
10                     NaN                          NaN   
11                     NaN                          NaN   
12                     NaN                          NaN   
13                     NaN                          NaN   
14                     NaN                          NaN   
15                     NaN                          NaN 

In [ ]:
# Adım 5: Yapılan tüm işlemleri fonksiyonlaştırınız.
def prepare_data(movie_path, rating_path):
    # Veri setlerini yükle
    movie = pd.read_csv(movie_path)
    rating = pd.read_csv(rating_path)
    
    # Rating ve Movie veri setlerini birleştir
    merged_df = pd.merge(rating, movie, on='movieId')
    
    # Toplam oy kullanılma sayısı 1000'in altında olan filmleri filtrele
    movie_counts = merged_df['title'].value_counts()
    popular_movies = movie_counts[movie_counts >= 1000].index
    filtered_df = merged_df[merged_df['title'].isin(popular_movies)]
    
    # UserID'lerin sütunlarda film isimlerinin ve değer olarak ratinglerin bulunduğu DataFrame için pivot table oluştur
    pivot_df = filtered_df.pivot_table(index='userId', columns='title', values='rating')
    
    return pivot_df

# Fonksiyonu kullanarak veri hazırlama
movie_file_path = "/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/movie.csv"
rating_file_path = "/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/rating.csv"
pivot_table = prepare_data(movie_file_path, rating_file_path)
print(pivot_table.head())


In [8]:
# Görev 2: Öneri Yapılacak Kullanıcının İzlediği Filmlerin Belirlenmesi
import numpy as np
# Adım1: Rastgelebirkullanıcıid’siseçiniz.
random_user_id = np.random.choice(pivot_df.index) 
# Adım 2: Seçilen kullanıcıya ait gözlem birimlerinden oluşan random_user_df adında yeni bir dataframe oluşturunuz. 
random_user_df = pivot_df[pivot_df.index == random_user_id]
# Adım 3: Seçilen kullanıcıların oy kullandığı filmleri movies_watched adında bir listeye atayınız
movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()

print(f"Seçilen Kullanıcı ID: {random_user_id}")
print("Kullanıcının İzlediği Filmler:")
print(movies_watched)

Seçilen Kullanıcı ID: 108940
Kullanıcının İzlediği Filmler:
['After Hours (1985)', 'Air America (1990)', 'As Good as It Gets (1997)', 'Bad Santa (2003)', 'Big (1988)', 'Big Daddy (1999)', 'Big Easy, The (1987)', 'Big Fish (2003)', 'Big Lebowski, The (1998)', "Big Momma's House (2000)", 'Big Trouble in Little China (1986)', 'Black Hawk Down (2001)', 'Black Rain (1989)', 'Casper (1995)', 'Cast Away (2000)', 'Copycat (1995)', 'Few Good Men, A (1992)', 'Field of Dreams (1989)', 'Fight Club (1999)', 'Finding Nemo (2003)', 'Goodfellas (1990)', 'Hidalgo (2004)', 'High Fidelity (2000)', 'House of Sand and Fog (2003)', 'Jackie Brown (1997)', 'Johnny Mnemonic (1995)', 'Life Is Beautiful (La Vita è bella) (1997)', 'Lost Boys, The (1987)', 'Meet Joe Black (1998)', 'Men in Black (a.k.a. MIB) (1997)', 'Monster (2003)', "Monster's Ball (2001)", 'Monsters, Inc. (2001)', 'Mummy, The (1999)', 'My Cousin Vinny (1992)', 'Name of the Rose, The (Name der Rose, Der) (1986)', 'Natural Born Killers (1994)', 'O

In [9]:
# Görev 2: Öneri Yapılacak Kullanıcının İzlediği Filmlerin Belirlenmesi
import numpy as np
# Adım1: Rastgelebirkullanıcıid’siseçiniz.
random_user_id = np.random.choice(pivot_df.index) 
# Adım 2: Seçilen kullanıcıya ait gözlem birimlerinden oluşan random_user_df adında yeni bir dataframe oluşturunuz. 
random_user_df = pivot_df[pivot_df.index == random_user_id]
# Adım 3: Seçilen kullanıcıların oy kullandığı filmleri movies_watched adında bir listeye atayınız
movies_watched = random_user_df.columns[random_user_df.notna().any()].tolist()

print(f"Seçilen Kullanıcı ID: {random_user_id}")
print("Kullanıcının İzlediği Filmler:")
print(movies_watched)

Seçilen Kullanıcı ID: 57399
Kullanıcının İzlediği Filmler:
["Amelie (Fabuleux destin d'Amélie Poulain, Le) (2001)", 'American Beauty (1999)', 'Blade (1998)', 'Blow (2001)', "Boys Don't Cry (1999)", 'Brokeback Mountain (2005)', 'Clockwork Orange, A (1971)', 'Crocodile Dundee (1986)', 'Crow, The (1994)', 'Daredevil (2003)', 'Donnie Darko (2001)', 'Dragon: The Bruce Lee Story (1993)', 'Dreamers, The (2003)', 'Edward Scissorhands (1990)', 'EuroTrip (2004)', 'Everything Is Illuminated (2005)', 'Evita (1996)', 'Fight Club (1999)', 'Four Rooms (1995)', 'Grindhouse (2007)', 'Importance of Being Earnest, The (2002)', 'Jingle All the Way (1996)', 'Kill Bill: Vol. 1 (2003)', 'Kill Bill: Vol. 2 (2004)', 'Lara Croft Tomb Raider: The Cradle of Life (2003)', 'Lara Croft: Tomb Raider (2001)', 'Lord of War (2005)', 'Lord of the Rings, The (1978)', 'Mr. & Mrs. Smith (2005)', 'Old Boy (2003)', 'Pet Sematary (1989)', 'Pirates of Silicon Valley (1999)', 'Prince of Egypt, The (1998)', 'Resident Evil (2002)'

In [10]:
# Görev 3: Aynı Filmleri İzleyen Diğer Kullanıcıların Verisine ve Id'lerine Erişilmesi
# Adım 1: Seçilen kullanıcının izlediği fimlere ait sutunları user_movie_df'ten seçiniz ve movies_watched_df adında yeni bir dataframe oluşturunuz. 
movies_watched_df = pivot_df.loc[:, movies_watched]
# Adım 2: Her bir kullancının seçili user'in izlediği filmlerin kaçını izlediğini bilgisini taşıyan user_movie_count adında yeni bir dataframe oluşturunuz.
user_movie_count = movies_watched_df.notna().sum(axis=1).reset_index()
user_movie_count.columns = ['userId', 'movies_watched_count']
# Adım 3: Seçilen kullanıcının oy verdiği filmlerin yüzde 60 ve üstünü izleyenlerin kullanıcı id’lerinden users_same_movies adında bir liste oluşturunuz.
threshold = len(movies_watched) * 0.60
users_same_movies = user_movie_count[user_movie_count['movies_watched_count'] >= threshold]['userId'].tolist()

In [11]:
print(f"Seçilen kullanıcı ID: {random_user_id}")
print("Benzer filmleri izleyen kullanıcılar:", users_same_movies)

Seçilen kullanıcı ID: 57399
Benzer filmleri izleyen kullanıcılar: [586, 768, 982, 1200, 1244, 1419, 1507, 1516, 2397, 3318, 3629, 3664, 3907, 4358, 4600, 4769, 4967, 5157, 5303, 5314, 6373, 6719, 6976, 7201, 7699, 7784, 7828, 8405, 8449, 9544, 9930, 10012, 10292, 10339, 10514, 10616, 10678, 10916, 11434, 11517, 11900, 12131, 12158, 12793, 13064, 13069, 13338, 13718, 13964, 14280, 14506, 14512, 15486, 15585, 15616, 15701, 15838, 16785, 16865, 16938, 17163, 17635, 17640, 18133, 18138, 18191, 18390, 18611, 18628, 18706, 19711, 20028, 22122, 22337, 22901, 23039, 23173, 23180, 24143, 24219, 24661, 24688, 24800, 24830, 25092, 25101, 25302, 25411, 25511, 25676, 25978, 26062, 26277, 27053, 27469, 27531, 28398, 28529, 28906, 29037, 29056, 29334, 29656, 29879, 30507, 31122, 32094, 32161, 32344, 32514, 32671, 33082, 33349, 33376, 33624, 33690, 33818, 34103, 34127, 34187, 34402, 34440, 34576, 34587, 34651, 34700, 34899, 35128, 35834, 37253, 37264, 37823, 38082, 38251, 38988, 39151, 39181, 40008, 4

In [12]:
# Görev 4: Öneri Yapılacak Kullanıcı ile En Benzer Kullanıcıların Belirlenmesi
# Adım 1: user_same_movies listesi içerisindeki seçili user ile benzerlik gösteren kullanıcıların id’lerinin bulunacağı şekilde movies_watched_df dataframe’ini filtreleyiniz.
filtered_users_df = movies_watched_df[movies_watched_df.index.isin(users_same_movies)]

# Adım 2: Kullanıcıların birbirleri ile olan korelasyonlarının bulunacağı yeni bir corr_df dataframe’i oluşturunuz.
corr_df = filtered_users_df.T.corr()
# Adım 3: Seçili kullanıcı ile yüksek korelasyona sahip (0.65’in üzerinde olan) kullanıcıları filtreleyerek top_users adında yeni bir dataframe oluşturunuz.
high_corr_users = corr_df[random_user_id][corr_df[random_user_id] > 0.65].index
top_users = corr_df.loc[high_corr_users]
# Adım 4: top_users dataframe’ine rating veri seti ile merge ediniz.
top_users_df = pd.merge(top_users, rating, left_index=True, right_on='userId')
top_users_df.head()

,586,768,982,1200,1244,1419,1507,1516,2397,3318,...,137277,137343,137686,137839,138208,138254,userId,movieId,rating,timestamp
1928568,0.814152,0.349332,0.730875,0.510786,0.316187,0.680264,0.43745,0.118146,0.038936,0.444897,...,0.729929,0.342145,0.349847,0.600111,0.586243,0.576295,13064,1,4.0,2012-02-07 23:07:03
1928569,0.814152,0.349332,0.730875,0.510786,0.316187,0.680264,0.43745,0.118146,0.038936,0.444897,...,0.729929,0.342145,0.349847,0.600111,0.586243,0.576295,13064,2,2.0,2012-02-07 23:19:22
1928570,0.814152,0.349332,0.730875,0.510786,0.316187,0.680264,0.43745,0.118146,0.038936,0.444897,...,0.729929,0.342145,0.349847,0.600111,0.586243,0.576295,13064,3,2.0,2012-02-07 23:34:30
1928571,0.814152,0.349332,0.730875,0.510786,0.316187,0.680264,0.43745,0.118146,0.038936,0.444897,...,0.729929,0.342145,0.349847,0.600111,0.586243,0.576295,13064,5,2.0,2012-02-07 23:32:55
1928572,0.814152,0.349332,0.730875,0.510786,0.316187,0.680264,0.43745,0.118146,0.038936,0.444897,...,0.729929,0.342145,0.349847,0.600111,0.586243,0.576295,13064,6,2.5,2012-02-07 23:15:11


In [18]:
# Görev 5: Weighted Average Recommendation Score'un Hesaplanması ve İlk 5 Filmin Tutulması
# Adım1: Her  bir kullanıcının corr ve rating değerlerinin çarpımından oluşan weighted_rating adında yeni bir değişken oluşturunuz.
top_users_df['correlation'] = top_users_df['userId'].map(lambda x: corr_df.loc[random_user_id, x])
top_users_df['weighted_rating'] = top_users_df['correlation'] * top_users_df['rating']

# Adım 2: Film id’si ve her bir filme ait tüm kullanıcıların weighted rating’lerinin ortalama değerini içeren recommendation_df adında yeni bir dataframe oluşturunuz.
recommendation_df = top_users_df.groupby('movieId').agg({'weighted_rating': 'mean'}).reset_index()

# Adım 3: recommendation_df içerisinde weighted rating'i 3.5'ten büyük olan filmleri seçiniz ve weighted rating’e göre sıralayınız.
recommended_movies = recommendation_df[recommendation_df['weighted_rating'] > 3.5]
recommended_movies = recommended_movies.sort_values('weighted_rating', ascending=False)


# Adım 4: movie veri setinden film isimlerini getiriniz ve tavsiye edilecek ilk 5 filmi seçiniz.
recommended_movies = recommended_movies.merge(movie[['movieId', 'title']], on='movieId', how='left')


In [19]:
recommended_movies.head(5)

,movieId,weighted_rating,title
0,3067,3.634189,Women on the Verge of a Nervous Breakdown (Muj...
1,3011,3.634189,"They Shoot Horses, Don't They? (1969)"
2,906,3.634189,Gaslight (1944)
3,2612,3.634189,Mildred Pierce (1945)
4,6639,3.634189,Wait Until Dark (1967)


In [14]:
######################## Item Based Recommendation ######################## 

In [4]:
import pandas as pd
import numpy as np

# Adım 1: movie ve rating veri setlerini okutunuz.
movie = pd.read_csv('/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/movie.csv')
rating = pd.read_csv('/Users/burcakaydin/PycharmProjects/pythonProject/recommender_systems/CS2_HybridRecommender/datasets/rating.csv')

# Adım 2: Rating veri setine filmlerin isimlerini ve türünü ekleyiniz.
df = pd.merge(rating, movie, on='movieId')

# Adım 3: user_movie_df oluşturmak için pivot tablo yapısını kullanınız.
# Toplam oy kullanılma sayısı 1000'in altında olan filmleri çıkartınız.
movie_counts = df['title'].value_counts()
popular_movies = movie_counts[movie_counts >= 1000].index
filtered_df = df[df['title'].isin(popular_movies)]

# Pivot tablo oluşturun
user_movie_df = filtered_df.pivot_table(index='userId', columns='title', values='rating')

# Adım 4: Kullanıcının en son izlediği ve en yüksek puan verdiği filmi bulma
user = 108170
user_ratings = rating[rating['userId'] == user]
user_high_ratings = user_ratings[user_ratings['rating'] == 5]
latest_high_rating = user_high_ratings.sort_values(by='timestamp', ascending=False).iloc[0]
selected_movie_id = latest_high_rating['movieId']
selected_movie_title = movie[movie["movieId"] == selected_movie_id]["title"].values[0]

# Adım 5: Seçilen filmle diğer filmlerin korelasyonunu bulma
selected_movie_ratings = user_movie_df[selected_movie_title]
correlation_with_selected_movie = user_movie_df.corrwith(selected_movie_ratings)
correlation_df = correlation_with_selected_movie.dropna().sort_values(ascending=False).to_frame()
correlation_df.columns = ['correlation']

# Adım 6: Seçilen film’in kendisi haricinde ilk 5 film’i öneri olarak veriniz.
def item_based_recommender(movie_name, user_movie_df):
    movie_series = user_movie_df[movie_name]
    return user_movie_df.corrwith(movie_series).sort_values(ascending=False).head(10)

# İlk 5 filmi öneri olarak verin (0'da filmin kendisi var, onu dışarda bırakın)
movies_from_item_based = item_based_recommender(selected_movie_title, user_movie_df)
recommended_movies = movies_from_item_based[1:6].index

# Önerilen filmleri yazdır
print(f"Kullanıcı ID: {user}")
print(f"En son izlediği ve en yüksek puan verdiği film: {selected_movie_title}")
print("Önerilen Filmler:")
print(recommended_movies)

Kullanıcı ID: 108170
En son izlediği ve en yüksek puan verdiği film: Wild at Heart (1990)
Önerilen Filmler:
Index(['My Science Project (1985)', 'Mediterraneo (1991)',
       'Old Man and the Sea, The (1958)',
       'National Lampoon's Senior Trip (1995)', 'Clockwatchers (1997)'],
      dtype='object', name='title')
